# CSV 8개 조인해서 master_df 만들기

**조인 구조**

- `olist_orders_dataset` (중심 테이블)
  - `order_id` → `order_items`, `order_payments`, `order_reviews`
  - `customer_id` → `customers`
- `order_items`
  - `product_id` → `products`
  - `seller_id` → `sellers`
- `customers`, `sellers` → 각자의 zip 컬럼으로 `geolocation` 에 연결

**grain(행 단위):** `order_items` 기준 — 한 주문에 상품이 여러 개면 행이 여러 개가 됩니다.
(payments / reviews 도 주문당 여러 건일 수 있어 행이 늘어날 수 있습니다 — 마지막 셀에서 진단 출력으로 확인)

In [1]:
import pandas as pd
from pathlib import Path

# CSV들이 이 노트북과 같은 폴더에 있음
DATA_DIR = Path.cwd()

orders    = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
reviews   = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
payments  = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
items     = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
products  = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers   = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
geo       = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")

for name, df in [
    ("orders", orders), ("reviews", reviews), ("payments", payments),
    ("items", items), ("customers", customers), ("products", products),
    ("sellers", sellers), ("geolocation", geo),
]:
    print(f"{name:12s} {df.shape}")


orders       (99441, 8)
reviews      (99224, 7)
payments     (103886, 5)
items        (112650, 7)
customers    (99441, 5)
products     (32951, 9)
sellers      (3095, 4)
geolocation  (1000163, 5)


In [2]:
# --- 1) geolocation 전처리 ---------------------------------------------------
# geolocation 은 zip_code_prefix 하나에 여러 좌표 행이 들어있음(중복).
# 그대로 조인하면 행이 폭발적으로 늘어나므로 zip별 대표값 1개로 집계한다.
geo_dedup = (
    geo.groupby("geolocation_zip_code_prefix", as_index=False)
       .agg(lat=("geolocation_lat", "mean"),
            lng=("geolocation_lng", "mean"),
            geo_city=("geolocation_city", "first"),
            geo_state=("geolocation_state", "first"))
)

# customers ← geolocation (customer_zip_code_prefix 기준)
cust_geo = geo_dedup.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "lat": "customer_geo_lat", "lng": "customer_geo_lng",
    "geo_city": "customer_geo_city", "geo_state": "customer_geo_state",
})
customers = customers.merge(cust_geo, on="customer_zip_code_prefix", how="left")

# sellers ← geolocation (seller_zip_code_prefix 기준)
sell_geo = geo_dedup.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "lat": "seller_geo_lat", "lng": "seller_geo_lng",
    "geo_city": "seller_geo_city", "geo_state": "seller_geo_state",
})
sellers = sellers.merge(sell_geo, on="seller_zip_code_prefix", how="left")

print("customers + geo:", customers.shape)
print("sellers   + geo:", sellers.shape)


customers + geo: (99441, 9)
sellers   + geo: (3095, 8)


In [3]:
# --- 2) order_items 에 products / sellers 붙이기 ------------------------------
# 각 상품 행에 상품 정보(product_id)와 판매자 정보(seller_id)를 1:1로 결합
items_full = (
    items.merge(products, on="product_id", how="left")
         .merge(sellers,  on="seller_id",  how="left")
)
print("items_full:", items_full.shape)


items_full: (112650, 22)


In [4]:
# --- 3) orders 를 중심으로 전체 결합 -----------------------------------------
# 모두 left join: orders 의 모든 주문을 살리고 부속 정보를 붙인다.
master_df = (
    orders
    .merge(customers,  on="customer_id", how="left")   # 주문 → 고객(+고객 지역)
    .merge(items_full, on="order_id",    how="left")   # 주문 → 상품행(+상품/판매자) ← grain 결정
    .merge(payments,   on="order_id",    how="left")   # 주문 → 결제
    .merge(reviews,    on="order_id",    how="left")   # 주문 → 리뷰
)

print("master_df:", master_df.shape)
master_df.head()


master_df: (119143, 47)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58


In [5]:
# --- 4) 진단 & 저장 ----------------------------------------------------------
print("최종 행/열 :", master_df.shape)
print("고유 주문 수:", master_df["order_id"].nunique(), "(원본 orders:", len(orders), ")")
print()
# 행이 늘어난 이유(주문당 여러 상품/결제/리뷰) 확인
print("주문당 평균 행 수:", round(len(master_df) / master_df["order_id"].nunique(), 3))
print("결측치 상위 컬럼:")
print(master_df.isna().mean().sort_values(ascending=False).head(10).round(3))

master_df.to_csv(DATA_DIR / "master_df.csv", index=False)
print("\n저장 완료 ->", DATA_DIR / "master_df.csv")


최종 행/열 : (119143, 47)
고유 주문 수: 99441 (원본 orders: 99441 )

주문당 평균 행 수: 1.198
결측치 상위 컬럼:
review_comment_title             0.883
review_comment_message           0.578
order_delivered_customer_date    0.029
product_photos_qty               0.021
product_category_name            0.021
product_name_lenght              0.021
product_description_lenght       0.021
order_delivered_carrier_date     0.018
seller_geo_lat                   0.009
seller_geo_lng                   0.009
dtype: float64

저장 완료 -> d:\user_dayeon\Sparta_Camp\team-oldest-olist-analysis\notebooks\Funnel_Cohort\DY\master_df.csv
